In [20]:
import kagglehub
import os
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import torch.nn as nn
import pandas as pd
from tqdm import tqdm
from src.model import HateSpeechClassifier
from src.utils import HateSpeechDataset, clean_text, collate_fn
from transformers import DistilBertTokenizerFast


In [21]:
model = HateSpeechClassifier()
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
if torch.cuda.is_available():
    model.load_state_dict(torch.load('checkpoints/hate_speech_model_weights.pth'))
else:
    model.load_state_dict(torch.load('checkpoints/hate_speech_model_weights.pth', map_location=torch.device('cpu')))

In [25]:
path = kagglehub.dataset_download("waalbannyantudre/hate-speech-detection-curated-dataset")
hate_speech_data = pd.read_csv(os.path.join(path, "HateSpeechDatasetBalanced.csv"))

_, sample_data = train_test_split(hate_speech_data, test_size=0.0003, random_state=42, shuffle=True, stratify=hate_speech_data['Label'])
sample_data['Cleaned_Content'] = sample_data['Content'].apply(clean_text)
print(sample_data['Label'].value_counts())
print(f"Number of samples for demo: {len(sample_data)}")

                                                  Content  Label  \
252854  magnificat for you sometimes please consider w...      0   
536847  or go to a stupid games with the terrorist who...      1   
425605  thanks for your hard work on wikipedia specifi...      0   
277588  need your help would it be possible to chat wi...      0   
231198  less than five editors even contributed to the...      0   

                                          Cleaned_Content  
252854  magnificat for you sometimes please consider w...  
536847  or go to a stupid games with the terrorist who...  
425605  thanks for your hard work on wikipedia specifi...  
277588  need your help would it be possible to chat wi...  
231198  less than five editors even contributed to the...  
Label
0    109
1    109
Name: count, dtype: int64
Number of samples for demo: 218


In [26]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sample_dataset = HateSpeechDataset(tokenizer=tokenizer, data=sample_data)
batch_size = 16
dataloader= DataLoader(sample_dataset,
                              batch_size=batch_size,
                              shuffle=True,
                              num_workers=2,
                              collate_fn=collate_fn,
                              pin_memory=True)
all_predictions = []
all_probabilities = []
all_texts = []
all_labels = []
model.to(device)

HateSpeechClassifier(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): L

In [27]:
with torch.no_grad():
    for i, batch in enumerate(tqdm(dataloader)):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        attention_mask = attention_mask.squeeze(1)
        batch_texts = batch['text']
        batch_labels = batch['labels'].to(device)
        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probabilities = torch.sigmoid(outputs.squeeze())
        predictions = (probabilities > 0.5).long()
        
        # Collect results
        all_predictions.extend(predictions.cpu().tolist())
        all_probabilities.extend(probabilities.cpu().tolist())
        all_texts.extend(batch_texts)
        all_labels.extend(batch_labels.cpu().tolist())

  0%|          | 0/14 [00:00<?, ?it/s]c:\Users\gauma\deep_learning_au25\Final_Proj\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
100%|██████████| 14/14 [00:36<00:00,  2.64s/it]


In [30]:
results_df = pd.DataFrame({
    'text': all_texts,
    'predicted_class': all_predictions,
    'true_class': all_labels,
    'probability': all_probabilities
})
results_df.to_csv('results/model_predictions.csv', index=False)
print("\nResults saved to 'model_predictions.csv'")


Results saved to 'model_predictions.csv'
